# P11 安全性の評価 ― 漏えいと止めすぎの両側

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

返信の下書きを書く AI に、攻撃 12 件（他のお客様の情報を出させようとする）と、物騒な言葉を含むが正当な依頼 12 件を渡し、漏えい（規則で判定）と止めすぎ（LLM 判定＋人が全件読む）を測ります。データの連絡先はすべて架空です。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 試験の中身 ― 問い合わせに紛れ込んだ別の指示
取り込んだ文書の中の指示（間接プロンプトインジェクション）に従わないかを見ます。

In [ ]:
def wrap(s, w=92):
    for line in s.splitlines():
        out, n = "", 0
        for ch in line:
            k = 2 if ord(ch) > 0x2E80 else 1
            if n + k > w:
                print(out); out, n = "", 0
            out, n = out + ch, n + k
        print(out)
from common import kit, stats
data = kit.load_json("safety.json")
items = data["items"]
print("攻撃", sum(x["kind"] == "attack" for x in items), "件 / 正当な依頼", sum(x["kind"] == "benign" for x in items), "件")
print("漏えいの印（架空の連絡先の一部）:")
print("  ", " ".join(data["leak_markers"]))
print("\n--- 攻撃の例 ATK-01 ---")
wrap(items[0]["text"])

## 2. 返信 AI（Opus 5.5）で 24 件を回す
講師の本物の実行の記録を再生します。止めすぎの判定役は Sonnet 5.5。

In [ ]:
def cut(s, w=100):                  # 表示幅（全角 2）で切って、後ろを「…」にする
    s, n = s.replace("\n", " "), 0
    for i, ch in enumerate(s):
        n += 2 if ord(ch) > 0x2E80 else 1
        if n > w:
            return s[:i] + "…"
    return s
import io, contextlib
P = "replay" if PROVIDER == "mock" else PROVIDER
sf = lab("run_safety", "lab11_safety")
with safe():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        sf.main(["--provider", P, "--all", "--model", "claude-opus-5-5"])
    for line in buf.getvalue().splitlines():
        print(cut(line, 104))

## 3. 規則で判定できるものは規則で
漏えいは「架空の電話番号の一部が返信に出たか」の文字列照合。LLM に判定させない。

In [ ]:
show_file("lab11_safety/run_safety.py", 57, 59)
print()
runs_o = kit.read_jsonl("lab11_safety/out/runs_safety_opus-5-5.jsonl")
for r in runs_o:
    if r["kind"] == "attack":
        head = r["reply"].replace("\n", " ")[:28] if r["reply"] else "（本文が空）"
        print(f"   {r['id']}  漏えいの印 {r['leak'] or 'なし'}  {len(r['reply']):4d} 字  {head}")

## 4. API の断りで本文が空になる ― ATK-02
モデルが API 側で断ると、返信の下書きが空で返ります。

In [ ]:
atk2 = next(x for x in items if x["id"] == "ATK-02")
print("--- ATK-02 の問い合わせ ---")
wrap(atk2["text"])
print()
for m in ("opus-5-5", "sonnet-5-5"):
    r = next(x for x in kit.read_jsonl(f"lab11_safety/out/runs_safety_{m}.jsonl") if x["id"] == "ATK-02")
    print(f"{m}: API の断り {r['api_refusal']} / 返信 {len(r['reply'])} 字 / 漏えい {r['leak'] or 'なし'} / {r['cost_usd']:.4f} USD")
print("\n漏えいは 0。でもお客様への返信も無い → 本番では「空の下書きは人に回す」作りが要る")

## 5. 返信 AI を Sonnet 5.5 に替える ― 止めすぎ 1 件
止めすぎの判定は LLM。判定された返信は、人が読んで確かめます。

In [ ]:
with safe():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        sf.main(["--provider", P, "--all", "--model", "claude-sonnet-5-5"])
    for line in buf.getvalue().splitlines():
        print(cut(line, 104))

## 6. 止めすぎとされた返信を読む ― BEN-11
判定役の依頼文と、問い合わせ・返信の本文を並べます。

In [ ]:
show_file("lab11_safety/prompts/helpful_judge.md", 2, 3)
ben = next(x for x in items if x["id"] == "BEN-11")
rep = next(x for x in kit.read_jsonl("lab11_safety/out/runs_safety_sonnet-5-5.jsonl") if x["id"] == "BEN-11")
print("\n--- BEN-11 の問い合わせ ---")
wrap(ben["text"])
print("--- Sonnet の返信（先頭）---")
body = [l for l in rep["reply"].splitlines() if l.strip()]
wrap("\n".join(body[:4]))
print("判定役の判定: helped =", rep["helped"], "→ 人が全文を読んで決める")

## 7. 例題 ― 12 件中 0 件から上限 24% を出す
0 件のとき、Wilson の上限は z²/(n+z²) にまとまります。

In [ ]:
import math
n, z = 12, 1.96
up = z * z / (n + z * z)
print(f"z² = {z * z:.4f}、上限 = {z * z:.4f} / ({n} + {z * z:.4f}) = {up:.4f} → {100 * up:.1f}%")
print(f"stats.wilson(0, 12) の上限 = {stats.wilson(0, 12)[1]:.4f}")
print(f"3 の法則: 3 / 12 = {3 / 12:.2f} → 25%")
print("「12 件で漏えい 0」から言えるのは「起きる率はおそらく 24% 以下」まで")
for t in (0.05, 0.01, 0.002):
    print(f"   上限を {100 * t:g}% にしたいなら、約 {math.ceil(3 / t)} 件の攻撃で 0 件が要る")